In [32]:
import duckdb
from pathlib import Path
import os

root = Path.cwd().parents[0]


conn = duckdb.connect()

In [45]:
p_flis_nsn_df = conn.execute(
    f"""
    SELECT *
    FROM read_parquet(
        '{root}/data/releases/SEP-2026/parquet/p_flis_nsn/*.parquet'
        )
    """
).df()

display(p_flis_nsn_df)



,FSC,NIIN,INC,ITEM_NAME,SOS
0,5960,001250026,00001,ELECTRON TUBE,B16
1,5960,001601779,00001,ELECTRON TUBE,
2,5960,007894457,00001,ELECTRON TUBE,
3,5960,010955002,00001,ELECTRON TUBE,NRP
4,5960,013103863,00001,ELECTRON TUBE,
...,...,...,...,...,...
17005577,0001,070001328,,,
17005578,0001,090000300,,,
17005579,0001,099848735,,,
17005580,4810,004884186,,,SMS


1. what does NIIN's with missing Souce of Supply columns mean?

In [27]:
v_h2_fsc_df = conn.execute(
    f"""
    SELECT *
    FROM read_parquet(
        '{root}/data/releases/SEP-2026/parquet/v_h2_fsc/*.parquet'
        )
    """
).df()

display(v_h2_fsc_df.head())

,FSC,FSC_TITLE,FSC_NOTES,FSC_INCLUSIONS,FSC_EXCLUSIONS
0,1005,"GUNS, THROUGH 30MM",,"INCLUDES MACHINE GUNS; BRUSHES, MACHINE GUN AN...","EXCLUDES TURRETS, AIRCRAFT."
1,1010,"GUNS, OVER 30MM UP TO 75MM",,INCLUDES BREECH MECHANISMS; MOUNTS; GRENADE LA...,
2,1015,"GUNS, 75MM THROUGH 125MM",,INCLUDES BREECH MECHANISMS; MOUNTS; RAMMERS.,
3,1020,"GUNS, OVER 125MM THROUGH 150MM",,INCLUDES BREECH MECHANISMS; POWER DRIVES; GUN ...,
4,1025,"GUNS, OVER 150MM THROUGH 200MM",,INCLUDES FIRING PLATFORMS; MOUNTS; GUN SHIELDS.,


In [28]:
display(v_h2_fsc_df[v_h2_fsc_df["FSC"]=="5960"])

,FSC,FSC_TITLE,FSC_NOTES,FSC_INCLUSIONS,FSC_EXCLUSIONS
401,5960,ELECTRON TUBES AND ASSOCIATED HARDWARE,,INCLUDES RECTIFYING TUBES; PHOTOELECTRIC TUBES...,EXCLUDES TRANSISTORS; TUBE SOCKETS; X-RAY TUBE...


In [ ]:
v_flis_id_df = conn.execute(
    f"""
    SELECT *
    FROM read_parquet(
        '{root}/data/releases/SEP-2026/parquet/v_flis_identification/*.parquet'
        )
    """
).df()

display(v_flis_id_df.head())

,NIIN,INC,CRIT_CD,DMIL,DMIL_INT_CD,NIIN_ASGMT,PMIC,HMIC,HCC,IUID_INDICATOR,LST_KWN_SOS
0,000000012,77777,,B,3,10-JAN-22,U,N,,,N32
1,000000018,77777,,B,3,10-JAN-22,U,N,,,N32
2,000000019,77777,,B,3,10-JAN-22,U,N,,,N32
3,000000040,77777,,A,4,10-JAN-22,U,N,,,S9I
4,000000041,77777,,A,4,10-JAN-22,A,N,,,D9S


In [30]:
cage_status_type_df = conn.execute(
    f"""
    SELECT *
    FROM read_parquet(
        '{root}/data/releases/SEP-2026/parquet/v_cage_status_and_type/*.parquet'
        )
    """
).df()

display(cage_status_type_df.head())

,CAGE_CODE,STATUS,TYPE,PARENT_CAGE,BUS_SIZE,PRIMARY_BUSINESS,TYPE_OF_BUSINESS,WOMAN_OWNED
0,00000,H,A,,N,N,,N
1,00001,H,A,,N,N,,N
2,00002,H,F,,N,N,N,N
3,00003,H,A,,N,N,,N
4,00004,H,A,,N,N,,N


In [31]:
cage_address_df = conn.execute(
    f"""
    SELECT *
    FROM read_parquet(
        '{root}/data/releases/SEP-2026/parquet/v_cage_address/*.parquet'
        )
    """
).df()

display(cage_address_df.head())

,CAGE_CODE,COMPANY_NAME,CITY,STATE,ZIP,COUNTRY
0,00000,ORDNANCE CORPS,BATTLE CREEK,MI,49017,UNITED STATES
1,00001,A B MFG CO,BATTLE CREEK,MI,49015,UNITED STATES
2,00002,STOUGHTON YOUTH COMMISSION,STOUGHTON,MA,02072-2571,UNITED STATES
3,00003,A B C PRODUCTS CO INC,LOS ANGELES,CA,90000,UNITED STATES
4,00004,A B F CO,CHICAGO,IL,60600,UNITED STATES


In [34]:
characteristics_df = conn.execute(
    f"""
    SELECT *
    FROM read_parquet(
        '{root}/data/releases/SEP-2026/parquet/v_characteristics/*.parquet'
        )
        LIMIT 10
    """
).df()

display(characteristics_df.head())

,NIIN,MRC,REQUIREMENTS_STATEMENT,CLEAR_TEXT_REPLY
0,000000042,AGAV,END ITEM IDENTIFICATION,USED ON ALM F13 AIRCRAFT
1,000000045,AGAV,END ITEM IDENTIFICATION,USED ON ALM F13 AIRCRAFT
2,000000047,AGAV,END ITEM IDENTIFICATION,OH-58
3,000000047,AJJX,COMPONENT DOCUMENT ORIGIN,INDUSTRIAL
4,000000047,AJJW,COMPONENT QUANTITY,4


1. Some NIIN's don't have a `COMPONENT QUANTITY` row

In [38]:
characteristics_df[characteristics_df["NIIN"].isna()].head()

,NIIN,MRC,REQUIREMENTS_STATEMENT,CLEAR_TEXT_REPLY


In [36]:
part_df = conn.execute(
    f"""
    SELECT *
    FROM read_parquet(
        '{root}/data/releases/SEP-2026/parquet/v_flis_part/*.parquet'
        )
    """
).df()

display(part_df.head())

,NIIN,PART_NUMBER,CAGE_CODE,CAGE_STATUS,RNCC,RNVC
0,000000042,UK 60A890216,24039,A,3,2
1,000000045,UK 60A890D255,24039,A,3,9
2,000000045,JWROGJWOEGIOW,0CEN3,Y,5,2
3,000000047,M30294,8V613,A,3,2
4,000000047,M30294,02731,R,5,9


In [46]:
flis_part_merged_df = p_flis_nsn_df.merge(part_df,"left",on="NIIN")

In [47]:
print(len(p_flis_nsn_df))
print(len(part_df))
print(len(flis_part_merged_df))
flis_part_merged_df.head()

17005582
16586217
26368522


,FSC,NIIN,INC,ITEM_NAME,SOS,PART_NUMBER,CAGE_CODE,CAGE_STATUS,RNCC,RNVC
0,5960,001250026,00001,ELECTRON TUBE,B16,SM-A-717589,80063,A,3,2
1,5960,001250026,00001,ELECTRON TUBE,B16,Y-442,06980,A,5,2
2,5960,001601779,00001,ELECTRON TUBE,,67A48E1-2,30003,A,3,2
3,5960,001601779,00001,ELECTRON TUBE,,IP936AAXQ,80058,A,5,1
4,5960,007894457,00001,ELECTRON TUBE,,67A48E1-1,10001,A,3,2
